# Experiment 3 Final Defense Report Notebook

**Experiment:** FastMCP-Based Instructional Tool Selection Runtime  
**Purpose:** Evaluate whether a real MCP runtime supports accurate, efficient instructional tool selection, comparing an MCP-baseline, unconditioned RAG-MCP, and FSLSM-conditioned RAG-MCP.

This notebook uses only the final canonical `exp3_mcp_runtime` implementation and the completed full run `exp3_core_real_20260503_1`.

In [ ]:
from pathlib import Path
import json
import os
import sqlite3

LOCAL_CACHE = Path.cwd().resolve() / ".notebook-cache"
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE))
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiments" / "exp3_mcp_runtime" / "results" / "runs" / "exp3_core_real_20260503_1").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root with final Exp3 run")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_ID = "exp3_core_real_20260503_1"
RUN_DIR = PROJECT_ROOT / "experiments" / "exp3_mcp_runtime" / "results" / "runs" / RUN_ID
DB_PATH = RUN_DIR / "exp3_runtime_results.db"
METRICS_PATH = RUN_DIR / "exp3_runtime_metrics.json"
FIGURES_DIR = RUN_DIR / "final_defense_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Run dir:      {RUN_DIR}")
print(f"Figures dir:  {FIGURES_DIR}")

## 1. Load Final Exp3-Core Run

The full run contains `60 questions x 16 profiles x 3 conditions = 2,880` session rows. The conditions are:

- `S0`: MCP-baseline / prompt-bloat MCP baseline
- `S1a`: RAG-MCP unconditioned
- `S1b`: RAG-MCP + FSLSM

`S0` must not be described as a no-MCP StandardRAG baseline.

In [ ]:
metrics = json.loads(METRICS_PATH.read_text())
benchmark = metrics["benchmarks"]["exp3_core"]
condition_order = ["S0", "S1a", "S1b"]

con = sqlite3.connect(DB_PATH)
sessions = pd.read_sql_query(
    """
    SELECT session_id, benchmark, condition, question_id, question_type, profile_label,
           selected_tool_id, selected_tool_name, task_optimal_tool_id, task_tsa_hit,
           profile_optimal_tool_id, profile_tsa_hit, profile_eval_eligible,
           pts_delta, input_tokens, latency_ms, execution_success, created_at
    FROM exp3_runtime_sessions
    """,
    con,
)
con.close()

condition_rows = []
for condition in condition_order:
    vals = benchmark["conditions"][condition]
    condition_rows.append({"condition": condition, **vals})
condition_df = pd.DataFrame(condition_rows)
condition_df["condition"] = pd.Categorical(condition_df["condition"], categories=condition_order, ordered=True)
condition_df = condition_df.sort_values("condition")

overview = pd.DataFrame([
    {"Measure": "Run ID", "Value": RUN_ID},
    {"Measure": "Total rows", "Value": len(sessions)},
    {"Measure": "Questions", "Value": sessions["question_id"].nunique()},
    {"Measure": "Profiles", "Value": sessions["profile_label"].nunique()},
    {"Measure": "Rows per condition", "Value": int(len(sessions) / 3)},
    {"Measure": "Execution success overall", "Value": f"{sessions['execution_success'].mean():.3f}"},
])
display(overview)
display(condition_df[["condition", "n", "task_tsa", "profile_tsa_all", "profile_tsa_eligible", "pts", "execution_success_rate", "latency_ms", "grounded_tool_output_rate"]].style.format({
    "task_tsa": "{:.3f}",
    "profile_tsa_all": "{:.3f}",
    "profile_tsa_eligible": "{:.3f}",
    "pts": "{:.1f}",
    "execution_success_rate": "{:.3f}",
    "latency_ms": "{:.1f}",
    "grounded_tool_output_rate": "{:.3f}",
}))

## 2. Primary And Secondary Accuracy Metrics

Task-TSA is the primary tool-selection metric. Profile-TSA is a secondary personalization analysis and should not be merged with Task-TSA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
x = np.arange(len(condition_df))
width = 0.24
axes[0].bar(x - width, condition_df["task_tsa"], width, label="Task-TSA", color="#2f6f9f")
axes[0].bar(x, condition_df["profile_tsa_all"], width, label="Profile-TSA All", color="#4f8f46")
axes[0].bar(x + width, condition_df["profile_tsa_eligible"], width, label="Profile-TSA Eligible", color="#d27d2d")
axes[0].set_xticks(x)
axes[0].set_xticklabels(condition_df["condition"])
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Exp3 Tool Selection Accuracy")
axes[0].legend()

axes[1].bar(condition_df["condition"], condition_df["pts"], color=["#8b8f97", "#2f6f9f", "#4f8f46"])
axes[1].set_ylim(0, 100)
axes[1].set_ylabel("Prompt Token Savings (%)")
axes[1].set_title("Prompt Token Savings Relative to S0")
for i, v in enumerate(condition_df["pts"]):
    axes[1].text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=9)
fig.tight_layout()
out = FIGURES_DIR / "exp3_defense_accuracy_pts.png"
fig.savefig(out, bbox_inches="tight")
plt.show()
print(out)

## 3. Matched Paired Comparisons

The paired comparison is computed over matched question-profile sessions. The main result is that `S1a` has the highest Task-TSA, while `S1b` shows a positive secondary Profile-TSA Eligible gain over `S1a`.

In [ ]:
paired_rows = []
for name, vals in benchmark["paired"].items():
    paired_rows.append({"comparison": name, **vals})
paired_df = pd.DataFrame(paired_rows)
display(paired_df.style.format({
    "task_tsa_delta": "{:+.3f}",
    "profile_tsa_all_delta": "{:+.3f}",
    "profile_tsa_eligible_delta": "{:+.3f}",
    "pts_delta": "{:+.1f}",
    "latency_ms_delta": "{:+.1f}",
}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
metrics_to_plot = ["task_tsa_delta", "profile_tsa_eligible_delta"]
labels = ["Task-TSA delta", "Profile-TSA Eligible delta"]
x = np.arange(len(paired_df))
width = 0.35
for offset, metric, label, color in [(-width/2, metrics_to_plot[0], labels[0], "#2f6f9f"), (width/2, metrics_to_plot[1], labels[1], "#d27d2d")]:
    axes[0].bar(x + offset, paired_df[metric], width, label=label, color=color)
axes[0].axhline(0, color="#222222", linewidth=1)
axes[0].set_xticks(x)
axes[0].set_xticklabels(paired_df["comparison"], rotation=20, ha="right")
axes[0].set_ylabel("Matched accuracy delta")
axes[0].set_title("Matched Accuracy Deltas")
axes[0].legend()

axes[1].bar(paired_df["comparison"], paired_df["latency_ms_delta"], color=["#b85c5c" if v > 0 else "#4f8f46" for v in paired_df["latency_ms_delta"]])
axes[1].axhline(0, color="#222222", linewidth=1)
axes[1].set_xticks(np.arange(len(paired_df)))
axes[1].set_xticklabels(paired_df["comparison"], rotation=20, ha="right")
axes[1].set_ylabel("Latency delta (ms)")
axes[1].set_title("Matched Latency Deltas")
fig.tight_layout()
out = FIGURES_DIR / "exp3_defense_paired_deltas.png"
fig.savefig(out, bbox_inches="tight")
plt.show()
print(out)

## 4. Runtime Validity Metrics

All conditions achieved complete execution success. The RAG-MCP conditions reduce prompt-token overhead and are faster than the MCP-baseline in this run.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
axes[0].bar(condition_df["condition"], condition_df["execution_success_rate"], color="#4f8f46")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Execution Success")
axes[0].set_ylabel("Rate")

axes[1].bar(condition_df["condition"], condition_df["latency_ms"], color="#6b8fb3")
axes[1].set_title("Mean Latency")
axes[1].set_ylabel("ms")

axes[2].bar(condition_df["condition"], condition_df["grounded_tool_output_rate"], color="#d27d2d")
axes[2].set_ylim(0, 1.0)
axes[2].set_title("Grounded Output Rate")
axes[2].set_ylabel("Rate")
fig.suptitle("Exp3 Runtime Validity Metrics", y=1.04)
fig.tight_layout()
out = FIGURES_DIR / "exp3_defense_runtime_validity.png"
fig.savefig(out, bbox_inches="tight")
plt.show()
print(out)

## 5. Tool-Level Task-TSA

This heatmap shows which target tools are routed correctly under each condition. It helps explain aggregate Task-TSA differences and reveals whether failures are concentrated in particular tool families.

In [ ]:
tool_tsa = (
    sessions.groupby(["condition", "task_optimal_tool_id"], as_index=False)["task_tsa_hit"]
    .mean()
    .pivot(index="task_optimal_tool_id", columns="condition", values="task_tsa_hit")
    .reindex(columns=condition_order)
)

fig, ax = plt.subplots(figsize=(7.2, 7.5))
im = ax.imshow(tool_tsa.values, aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(tool_tsa.columns)))
ax.set_xticklabels(tool_tsa.columns)
ax.set_yticks(np.arange(len(tool_tsa.index)))
ax.set_yticklabels([f"Tool {int(t)}" for t in tool_tsa.index])
for i in range(tool_tsa.shape[0]):
    for j in range(tool_tsa.shape[1]):
        ax.text(j, i, f"{tool_tsa.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Task-TSA by Target Tool")
fig.colorbar(im, ax=ax, label="Task-TSA")
fig.tight_layout()
out = FIGURES_DIR / "exp3_defense_tool_level_tsa.png"
fig.savefig(out, bbox_inches="tight")
plt.show()
print(out)

## 6. Selected Tool Distribution

The selected-tool distribution provides a compact sanity check that the MCP selector is using the tool space broadly rather than collapsing to a single tool.

In [ ]:
dist = sessions.groupby(["condition", "selected_tool_id"]).size().reset_index(name="n")
pivot = dist.pivot(index="selected_tool_id", columns="condition", values="n").fillna(0).reindex(columns=condition_order)

fig, ax = plt.subplots(figsize=(10, 5.2))
x = np.arange(len(pivot.index))
width = 0.25
for idx, condition in enumerate(condition_order):
    ax.bar(x + (idx - 1) * width, pivot[condition], width, label=condition)
ax.set_xticks(x)
ax.set_xticklabels([f"{int(t)}" for t in pivot.index])
ax.set_xlabel("Selected tool id")
ax.set_ylabel("Selection count")
ax.set_title("Selected Tool Distribution by Condition")
ax.legend(title="Condition")
fig.tight_layout()
out = FIGURES_DIR / "exp3_defense_selected_tool_distribution.png"
fig.savefig(out, bbox_inches="tight")
plt.show()
print(out)

## 7. Defense-Ready Interpretation

Use the following points in the final report or oral defense:

1. **Runtime result:** Exp3 completed a full FastMCP-based run with 2,880 session rows and 100% execution success.
2. **Primary metric:** `S1a` achieved the highest Task-TSA, so the result does not support a broad claim that FSLSM conditioning improves task-oriented tool accuracy over unconditioned RAG-MCP.
3. **Efficiency result:** Both `S1a` and `S1b` preserve approximately 93.5% prompt token savings relative to the MCP-baseline.
4. **Secondary personalization result:** `S1b` improves over `S1a` on Profile-TSA Eligible by +0.094, indicating a limited personalization-specific benefit where profile-conditioned tool divergence is meaningful.
5. **Thesis framing:** The strongest Exp3 claim is efficient, valid MCP-based tool execution with a limited FSLSM personalization effect, not universal superiority of FSLSM-conditioned selection.

In [ ]:
figure_files = sorted(FIGURES_DIR.glob("*.png"))
figure_index = pd.DataFrame({
    "figure_file": [p.name for p in figure_files],
    "path": [str(p.relative_to(PROJECT_ROOT)) for p in figure_files],
})
display(Markdown("### Generated figure files"))
display(figure_index)